<a href="https://colab.research.google.com/github/NatalieAbel/Capstone-Project/blob/main/stable_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
from tqdm import tqdm
import diffusers
from diffusers import StableDiffusionXLPipeline
import os
import pandas as pd
import subprocess

from PIL import Image
from IPython.display import display
from transformers import AutoImageProcessor, AutoModel

import torchaudio
from transformers import AutoProcessor, HubertModel


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
#get sample of audiocaps dataset
df = pd.read_csv("audiocaps.csv")

#oversample since some YouTube clips fail
sample_df = df.sample(n=200, random_state=42).copy()
sample_df.to_csv("audiocaps_sample_200.csv", index=False)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its result

In [ ]:
pip install yt-dlp

In [ ]:
#generate audio of 200 sample dataset
sample_df = pd.read_csv("audiocaps_sample_200.csv")

audio_root = "audio"
temp_root = "temp_audio"
os.makedirs(audio_root, exist_ok=True)
os.makedirs(temp_root, exist_ok=True)

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    audiocap_id = str(row["audiocap_id"])
    youtube_id = str(row["youtube_id"])
    start_time = float(row["start_time"])

    final_wav_path = os.path.join(audio_root, f"{audiocap_id}.wav")
    if os.path.exists(final_wav_path):
        continue

    youtube_url = f"https://www.youtube.com/watch?v={youtube_id}"
    temp_template = os.path.join(temp_root, f"{youtube_id}.%(ext)s")

    #download best audio only
    download_cmd = [
        "yt-dlp",
        "-f", "bestaudio",
        "-o", temp_template,
        youtube_url
    ]
    subprocess.run(download_cmd, capture_output=True, text=True)

    downloaded_path = None
    for ext in ["m4a", "webm", "mp3", "opus"]:
        sample = os.path.join(temp_root, f"{youtube_id}.{ext}")
        if os.path.exists(sample):
            downloaded_path = sample
            break

    if downloaded_path is None:
        continue

    #trim to the 10-second AudioCaps clip
    ffmpeg_cmd = [
        "ffmpeg",
        "-y",
        "-ss", str(start_time),
        "-i", downloaded_path,
        "-t", "10",
        "-ac", "1",
        "-ar", "16000",
        final_wav_path
    ]
    subprocess.run(ffmpeg_cmd, capture_output=True, text=True)

    #delete temp full audio file to save space
    if os.path.exists(downloaded_path):
        os.remove(downloaded_path)

In [ ]:
#keep only rows whose audio exists and make 100-sample CSV
sample_df = pd.read_csv("audiocaps_sample_200.csv")

valid_rows = []

for _, row in sample_df.iterrows():
    audio_id = str(row["audiocap_id"])
    audio_path = os.path.join("audio", f"{audio_id}.wav")
    if os.path.exists(audio_path):
        valid_rows.append(row)

valid_df = pd.DataFrame(valid_rows)
valid_df.to_csv("audiocaps_sample_200_with_audio.csv", index=False)

print("Valid rows with audio:", len(valid_df))

final_df = valid_df.sample(n=100, random_state=42).copy()
final_df.to_csv("audiocaps_sample_100.csv", index=False)

print("Final sample size:", len(final_df))

In [ ]:
#generate large images for 100-sample subset

#load subset and stable diffusion pipeline
subset_df = pd.read_csv("audiocaps_sample_100.csv")

model_id = "stabilityai/stable-diffusion-xl-base-1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"

#initialize the pipeline
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()

for _, row in tqdm(subset_df.iterrows(),
    total = len(subset_df)):
      caption = row["caption"]
      audio_id = str(row["audiocap_id"])
      output_dir = "images/stable_diffusion/"
      model_dir = os.path.join(output_dir, audio_id)
      if not os.path.isdir(model_dir):
        os.makedirs(model_dir, exist_ok=True)
      files = [f for f in os.listdir(model_dir)]
      prompt = 'Generate an image of: ' + caption
      if len(files) >= 1:
        continue
      else:

        print(prompt, len(files))
        with torch.no_grad():
          generator = torch.manual_seed(42)
          image = pipe(prompt, generator=generator).images[0]  #generates one frame
          image.save(os.path.join(model_dir, "0.png"))

In [ ]:
#preview generated images
subset_df = pd.read_csv("audiocaps_sample_100.csv")

for _, row in subset_df.iterrows():
    audio_id = str(row["audiocap_id"])
    img_path = f"images/stable_diffusion_xl/{audio_id}/0.png"

    if os.path.exists(img_path):
        print("Audio ID:", audio_id)
        print("Caption:", row["caption"])
        img = Image.open(img_path)
        img = img.resize((150, 150))
        display(img)
        print("-" * 60)

In [ ]:
#extract DINOv2 (large) embeddings from images

device = "cuda" if torch.cuda.is_available() else "cpu"

#load images and DINOv2 model
subset_df = pd.read_csv("audiocaps_sample_100.csv")

image_root = "images/stable_diffusion_xl"
save_root = "images/image_embeddings/DINOV2_LARGE_SDXL"
os.makedirs(save_root, exist_ok=True)

model_name = "facebook/dinov2-large"

#load processor and model
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

#extract embeddings
for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
    audio_id = str(row["audiocap_id"])

    img_path = os.path.join(image_root, audio_id, "0.png")
    save_path = os.path.join(save_root, f"{audio_id}.npy")

    if not os.path.exists(img_path):
        print(f"Missing image for {audio_id}")
        continue

    if os.path.exists(save_path):
        continue

    image = Image.open(img_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embedding = cls_embedding.squeeze(0).cpu().numpy()

    np.save(save_path, embedding)

In [ ]:
#download image embeddings in zip file
!zip -r dinov2_large_embeddings_sdxl.zip images/image_embeddings/DINOV2_LARGE_SDXL
from google.colab import files
files.download("dinov2_large_embeddings_sdxl.zip")

In [ ]:
#extract DINOv2 (base) embeddings from images

device = "cuda" if torch.cuda.is_available() else "cpu"

#load images and DINOv2 model
subset_df = pd.read_csv("audiocaps_sample_100.csv")

image_root = "images/stable_diffusion_base"
save_root = "images/image_embeddings/DINOV2_BASE_SDXL"
os.makedirs(save_root, exist_ok=True)

model_name = "facebook/dinov2-base"

#load processor and model
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

#extract embeddings
for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
    audio_id = str(row["audiocap_id"])

    img_path = os.path.join(image_root, audio_id, "0.png")
    save_path = os.path.join(save_root, f"{audio_id}.npy")

    if not os.path.exists(img_path):
        print(f"Missing image for {audio_id}")
        continue

    if os.path.exists(save_path):
        continue

    image = Image.open(img_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embedding = cls_embedding.squeeze(0).cpu().numpy()

    np.save(save_path, embedding)

In [ ]:
#download image embeddings in zip file
!zip -r dinov2_large_embeddings_sdxl.zip images/image_embeddings/DINOV2_LARGE_SDXL
from google.colab import files
files.download("dinov2_base_embeddings_sdxl.zip")

In [ ]:
#extract HuBERT (large) embeddings from audio

device = "cuda" if torch.cuda.is_available() else "cpu"

#load audio and processor
subset_df = pd.read_csv("audiocaps_sample_100.csv")

audio_root = "audio"
save_root = "audio/audio_embeddings/HUBERT_LARGE"
os.makedirs(save_root, exist_ok=True)

model_name = "facebook/hubert-large-ls960-ft"

processor = AutoProcessor.from_pretrained(model_name)
model = HubertModel.from_pretrained(model_name).to(device)
model.eval()

#extract embeddings
for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
    audio_id = str(row["audiocap_id"])

    audio_path = os.path.join(audio_root, f"{audio_id}.wav")
    save_path = os.path.join(save_root, f"{audio_id}.npy")

    if not os.path.exists(audio_path):
        print(f"Missing audio for {audio_id}")
        continue

    if os.path.exists(save_path):
        continue

    waveform, sr = torchaudio.load(audio_path)

    waveform = waveform.mean(dim=0)

    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    inputs = processor(
        waveform,
        sampling_rate=16000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        hidden_states = outputs.last_hidden_state


        embedding = hidden_states.mean(dim=1).squeeze(0).cpu().numpy()

    np.save(save_path, embedding)

In [ ]:
#download audio embeddings in zip file
!zip -r hubert_large_audio_embeddings.zip audio/audio_embeddings/HUBERT_LARGE
from google.colab import files
files.download("hubert_large_audio_embeddings.zip")